In [ ]:
#Import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re

#Libraries used to download input file from email
import win32com.client
import os
import shutil

#Libraries used to open the files
import glob
from openpyxl import load_workbook

#Run access.ipynb
%run ../../access.ipynb

#### Delete old files from input folder

In [ ]:
def clear_download_folder(download_folder, ignore_files=None):
    if ignore_files is None:
        ignore_files = []

    # Check if the folder exists
    if os.path.exists(download_folder):
        # Loop through each file in the folder
        for filename in os.listdir(download_folder):
            # Skip files that are in the ignore list
            if filename in ignore_files:
                continue

            file_path = os.path.join(download_folder, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)  # Remove the file
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)  # Remove the directory
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')

# Define the download folder path
download_folder_path = r'../input'  # Specify the relative path

# Clear the download folder, ignoring 'read.md'
clear_download_folder(download_folder_path, ignore_files=['read.md'])

In [ ]:
#Define function that ensures files downloaded are not overwriten
def get_unique_file_path(directory, original_filename, new_name_base):
    _, extension = os.path.splitext(original_filename)
    new_filename = f"{new_name_base}{extension}"
    counter = 1
    while os.path.exists(os.path.join(directory, new_filename)):
        new_filename = f"{new_name_base}{counter}{extension}"
        counter += 1
    return os.path.join(directory, new_filename)

##### Define function that gets reports from email

In [ ]:
def download_attachment_from_email(subject_keywords, download_folder, patterns_dict, user_email, folder_name):
    # Initialize Outlook application and get the namespace for MAPI
    outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")
    # Access the specific user's inbox and the specific folder
    inbox = outlook.Folders.Item(user_email).Folders.Item(folder_name)
    
    # Retrieve messages and sort by received time in descending order
    messages = inbox.Items
    messages.Sort("[ReceivedTime]", True)
    
    # Loop through each message
    for message in messages:
        # Check if the subject contains any of the specified keywords
        if any(keyword.lower() in message.Subject.lower() for keyword in subject_keywords):
            # Access attachments in the message
            attachments = message.Attachments
            # Loop through each attachment
            for attachment in attachments:
                # Check each pattern and corresponding name in the dictionary
                for filename_pattern, new_name_base in patterns_dict.items():
                    if filename_pattern.lower() in attachment.FileName.lower():
                        # Ensure the download folder exists
                        if not os.path.exists(download_folder):
                            os.makedirs(download_folder)
                        
                        # Generate a unique file path for saving the attachment
                        unique_file_path = get_unique_file_path(download_folder, attachment.FileName, new_name_base)
                        absolute_unique_file_path = os.path.abspath(unique_file_path)
                        
                        # Print the path where the file will be saved (for debugging)
                        print(f"Saving attachment to: {absolute_unique_file_path}")
                        
                        # Save the attachment to the unique file path
                        attachment.SaveAsFile(absolute_unique_file_path)
                        # Removing return here allows the loop to continue
            # Consider breaking out of the loop if no more messages need processing
            break  # Stop after finding the most recent email with the subject keywords

#### Get files from email

In [ ]:
##################################################
#Get Ingram Reports (1 email / 2 files)
##################################################

# Define dictionaries that will get the file from email and rename them
patterns_dict = {
    "SO": "ingram_sales_report",
    "INV": "ingram_stock_report"
}

# Define the email subject keywords and the download folder
subject_keywords = ["Ventas e Inventario ASUS del"]
download_folder_path = r'../input'  # Specify the relative path

# Call the function to download the attachments
download_attachment_from_email(subject_keywords, download_folder_path, patterns_dict, user_email, country_reports_folder )


In [ ]:
##################################################
#Get Intcomex Reports (2 email / 2 files)
##################################################

# Get Sales Report

# Define the email subject keywords and the download folder
subject_keywords = ["INTX-W-ASU-POS"]
download_folder_path = r'../input'  # Specify the relative path

# Define dictionaries that will get the file from email and rename them
patterns_dict = {
    "INTX-Weekly-ASU-POS": "intcomex_sales_report"
}

# Call the function to download the attachments
download_attachment_from_email(subject_keywords, download_folder_path, patterns_dict, user_email, country_reports_folder )

# Get Stock Report

# Define the email subject keywords and the download folder
subject_keywords = ["INTX-W-ASU-INV"]
download_folder_path = r'../input'  # Specify the relative path

# Define dictionaries that will get the file from email and rename them
patterns_dict = {
    "INTX-Weekly-ASU-INV": "intcomex_stock_report"
}

# Call the function to download the attachments
download_attachment_from_email(subject_keywords, download_folder_path, patterns_dict, user_email, country_reports_folder )

In [ ]:
##################################################
#Get Nexsys Reports (1 email / 1 file)
##################################################

# Define the email subject keywords and the download folder
subject_keywords = ["REPORTES ASUS_NEXSYS"]
download_folder_path = r'../input'  # Specify the relative path

# Define dictionaries that will get the file from email and rename them
patterns_dict = {
    "weekly report": "nexsys_report"
}

# Call the function to download the attachments
download_attachment_from_email(subject_keywords, download_folder_path, patterns_dict, user_email, country_reports_folder )

In [ ]:
##################################################
#Get Tecnoglobal Reports (1 email / 1 file)
##################################################

# Define the email subject keywords and the download folder
subject_keywords = ["Reporte Ventas WK"]
download_folder_path = r'../input'  # Specify the relative path

# Define dictionaries that will get the file from email and rename them
patterns_dict = {
    "Reporte ASUS Computo": "tecnoglobal_report"
}

# Call the function to download the attachments
download_attachment_from_email(subject_keywords, download_folder_path, patterns_dict, user_email, country_reports_folder )

In [ ]:
##################################################
#Get PCFactory Reports (1 email / 2 files)
##################################################

# Define dictionaries that will get the file from email and rename them
patterns_dict = {
    "Ventas Asus": "pcfactory_sales_report",
    "Stock Asus": "pcfactory_stock_report"
}

# Define the email subject keywords and the download folder
subject_keywords = ["Reportes PCF"]
download_folder_path = r'../input'  # Specify the relative path

# Call the function to download the attachments
download_attachment_from_email(subject_keywords, download_folder_path, patterns_dict, user_email, country_reports_folder )
